In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import skforecast

print(skforecast.__version__)

c:\Users\Joaquin\Documents\GitHub\skforecast
0.24.0


In [2]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
from skforecast.datasets import fetch_dataset

# Modelling and Forecasting
# ==============================================================================
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.preprocessing import RollingFeatures
from sklearn.linear_model import Ridge

In [3]:
# Data download
# ==============================================================================
data = fetch_dataset(name='bike_sharing', raw=False, verbose=False)
data = data[['users', 'temp', 'hum', 'windspeed', 'holiday']]
data = data.loc['2011-04-01 00:00:00':'2012-10-20 23:00:00', :].copy()
# Split train-validation-test
# ==============================================================================
end_train = '2012-06-30 23:59:00'
end_validation = '2012-10-01 23:59:00'
data_train = data.loc[: end_train, :]
data_val   = data.loc[end_train:end_validation, :]
data_test  = data.loc[end_validation:, :]

print(f"Dates train      : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Dates validation : {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_val)})")
print(f"Dates test       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Dates train      : 2011-04-01 00:00:00 --- 2012-06-30 23:00:00  (n=10968)
Dates validation : 2012-07-01 00:00:00 --- 2012-10-01 23:00:00  (n=2232)
Dates test       : 2012-10-02 00:00:00 --- 2012-10-20 23:00:00  (n=456)


In [4]:
# Create and fit forecaster
# ==============================================================================
params = {
    "max_depth": 7,
    "n_estimators": 300,
    "learning_rate": 0.06,
    "verbose": -1,
    "random_state": 15926,
}
lags = [1, 2, 3, 23, 24, 25, 167, 168, 169]
window_features = RollingFeatures(stats=["mean"], window_sizes=24 * 3)

forecaster = ForecasterRecursive(
                 estimator     = LGBMRegressor(**params),
                 lags            = lags,
                 window_features = window_features,
             )


In [5]:
%%timeit

forecaster.fit(
    y    = data.loc[:end_validation, 'users'],
    exog = data.loc[:end_validation, ['temp', 'hum', 'windspeed', 'holiday']],
    store_in_sample_residuals = True
)

The slowest run took 4.13 times longer than the fastest. This could mean that an intermediate result is being cached.
1.45 s ± 612 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit

_ = forecaster.predict_bootstrapping(steps=len(data_test), exog = data_test[['temp', 'hum', 'windspeed', 'holiday']],  n_boot=250)

559 ms ± 54.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
# Create and fit forecaster
# ==============================================================================
params = {
    "max_depth": 7,
    "n_estimators": 300,
    "learning_rate": 0.06,
    "silent": True, 
    "allow_writing_files"  :False,
    "random_state": 15926
}
lags = [1, 2, 3, 23, 24, 25, 167, 168, 169]
window_features = RollingFeatures(stats=["mean"], window_sizes=24 * 3)

forecaster = ForecasterRecursive(
                 estimator     = CatBoostRegressor(**params),
                 lags            = lags,
                 window_features = window_features,
             )


In [8]:
%%timeit

forecaster.fit(
    y    = data.loc[:end_validation, 'users'],
    exog = data.loc[:end_validation, ['temp', 'hum', 'windspeed', 'holiday']],
    store_in_sample_residuals = True
)

1.65 s ± 56.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%%timeit

_ = forecaster.predict_bootstrapping(steps=len(data_test), exog = data_test[['temp', 'hum', 'windspeed', 'holiday']],  n_boot=250)

632 ms ± 20.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
# Create and fit forecaster
# ==============================================================================
params = {
    "max_depth": 7,
    "max_iter": 300,
    "learning_rate": 0.06,
    "random_state": 15926
}
lags = [1, 2, 3, 23, 24, 25, 167, 168, 169]
window_features = RollingFeatures(stats=["mean"], window_sizes=24 * 3)

forecaster = ForecasterRecursive(
                 estimator     = HistGradientBoostingRegressor(**params),
                 lags            = lags,
                 window_features = window_features,
             )


In [11]:
%%timeit

forecaster.fit(
    y    = data.loc[:end_validation, 'users'],
    exog = data.loc[:end_validation, ['temp', 'hum', 'windspeed', 'holiday']],
    store_in_sample_residuals = True
)

805 ms ± 49.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit

_ = forecaster.predict_bootstrapping(steps=len(data_test), exog = data_test[['temp', 'hum', 'windspeed', 'holiday']],  n_boot=250)

2.49 s ± 116 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
# Create and fit forecaster
# ==============================================================================
lags = [1, 2, 3, 23, 24, 25, 167, 168, 169]
window_features = RollingFeatures(stats=["mean"], window_sizes=24 * 3)

forecaster = ForecasterRecursive(
                 estimator     = Ridge(random_state=6718),
                 lags            = lags,
                 window_features = window_features,
             )

In [14]:
%%timeit

forecaster.fit(
    y    = data.loc[:end_validation, 'users'],
    exog = data.loc[:end_validation, ['temp', 'hum', 'windspeed', 'holiday']],
    store_in_sample_residuals = True
)

13.6 ms ± 464 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
%%timeit

_ = forecaster.predict_bootstrapping(steps=len(data_test), exog = data_test[['temp', 'hum', 'windspeed', 'holiday']],  n_boot=250)

126 ms ± 3.78 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
